In [24]:
import numpy as np
import cv2
import glob
from google.colab.patches import cv2_imshow
from google.colab import drive

In [25]:
# 1. Montage de Google Drive
drive.mount('/content/drive')

# 2. Paramètres physiques (Olympus TG-6 et damier)
taille_carre_mm = 30.0
largeur_capteur_mm = 6.17
hauteur_capteur_mm = 4.55

colonnes_internes = 8
lignes_internes = 5

# Critères de terminaison
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 3. Préparation des points objets (Mis à l'échelle en mm dès le départ)
objp = np.zeros((lignes_internes * colonnes_internes, 3), np.float32)
objp[:, :2] = np.mgrid[0:colonnes_internes, 0:lignes_internes].T.reshape(-1, 2) * taille_carre_mm

# 4. Chargement des images
images = glob.glob('/content/drive/MyDrive/seq_data/*')
print(f"{len(images)} images trouvées dans le dossier.")

# Variables globales pour la suite
objpoints = []
imgpoints = []
images_valides = [] # Stocke uniquement les images où le damier est trouvé

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
11 images trouvées dans le dossier.


In [ ]:
for fname in images:
    img = cv2.imread(fname)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret == True:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)
        images_valides.append(fname)

        # Dessiner et afficher
        img_show = cv2.drawChessboardCorners(img.copy(), (colonnes_internes, lignes_internes), corners2, ret)
        print(f"Succès pour : {fname}")
        cv2_imshow(img_show)
    else:
        print(f"Échec de la détection pour : {fname}")

In [27]:
# Calibration
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

print("Erreur de reprojection (RMS) :", ret)
print("\nMatrice de la caméra (Intrinsèque) :\n", mtx)
print("\nCoefficients de distorsion :\n", dist)

# Extraction des caractéristiques physiques
fovx, fovy, focalLength, principalPoint, aspectRatio = cv2.calibrationMatrixValues(
    cameraMatrix=mtx,
    imageSize=gray.shape[::-1],
    apertureWidth=largeur_capteur_mm,
    apertureHeight=hauteur_capteur_mm
)

focale_metre = focalLength / 1000.0

print(f"\nChamp de vision X (FOV X) : {fovx:.2f} degrés")
print(f"Champ de vision Y (FOV Y) : {fovy:.2f} degrés")
print(f"\n===> Focale de la caméra : {focale_metre:.6f} mètres <===")
print(f"Point principal (Centre optique) : {principalPoint}")

Erreur de reprojection (RMS) : 4.8113589471852745

Matrice de la caméra (Intrinsèque) :
 [[3.10939389e+03 0.00000000e+00 2.00531651e+03]
 [0.00000000e+00 3.10839028e+03 1.47355906e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Coefficients de distorsion :
 [[-2.30477705e-01  2.38488363e+00 -5.49698553e-03  3.80292843e-03
  -7.75769701e+00]]

Champ de vision X (FOV X) : 65.50 degrés
Champ de vision Y (FOV Y) : 51.52 degrés

===> Focale de la caméra : 0.004796 mètres <===
Point principal (Centre optique) : (3.0932007171420586, 2.23489790373902)


In [ ]:
# Définition de l'axe (3 cases de 30mm)
longueur_axe = 3 * taille_carre_mm
axis = np.float32([[longueur_axe,0,0], [0,longueur_axe,0], [0,0,-longueur_axe]]).reshape(-1,3)

def draw_axes(img, corners, imgpts):
    corner = tuple(corners[0].ravel().astype(int))
    img = cv2.line(img, corner, tuple(imgpts[0].ravel().astype(int)), (255,0,0), 5) # Axe X Bleu
    img = cv2.line(img, corner, tuple(imgpts[1].ravel().astype(int)), (0,255,0), 5) # Axe Y Vert
    img = cv2.line(img, corner, tuple(imgpts[2].ravel().astype(int)), (0,0,255), 5) # Axe Z Rouge
    return img

for fname in images_valides:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        ret_pnp, rvecs_img, tvecs_img = cv2.solvePnP(objp, corners2, mtx, dist)
        imgpts, jac = cv2.projectPoints(axis, rvecs_img, tvecs_img, mtx, dist)

        img_axes = draw_axes(img, corners2, imgpts)
        print(f"Repère 3D pour : {fname}")
        cv2_imshow(img_axes)

In [ ]:
# Définition du cube (3 cases de 30mm de côté)
taille_cube = 3 * taille_carre_mm
axis_cube = np.float32([
    [0, 0, 0], [0, taille_cube, 0], [taille_cube, taille_cube, 0], [taille_cube, 0, 0],
    [0, 0, -taille_cube], [0, taille_cube, -taille_cube], [taille_cube, taille_cube, -taille_cube], [taille_cube, 0, -taille_cube]
])

def draw_cube(img, corners, imgpts):
    imgpts = np.int32(imgpts).reshape(-1, 2)
    img = cv2.drawContours(img, [imgpts[:4]], -1, (0, 255, 0), -3) # Plancher vert
    for i, j in zip(range(4), range(4, 8)):
        img = cv2.line(img, tuple(imgpts[i]), tuple(imgpts[j]), (255, 0, 0), 3) # Piliers bleus
    img = cv2.drawContours(img, [imgpts[4:]], -1, (0, 0, 255), 3) # Toit rouge
    return img

for fname in images_valides:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (colonnes_internes, lignes_internes), None)

    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        ret_pnp, rvecs_img, tvecs_img = cv2.solvePnP(objp, corners2, mtx, dist)
        imgpts, jac = cv2.projectPoints(axis_cube, rvecs_img, tvecs_img, mtx, dist)

        img_cube = draw_cube(img, corners2, imgpts)
        print(f"Cube 3D pour : {fname}")
        cv2_imshow(img_cube)